In [ ]:
# %% Deep learning - Section 19.182
#    Code challenge 30: custom loss function
#
#    1) Start from code from video 19.180 (gaussian synthetic dataset)
#    2) Implement the three custom loss function shown in the video (in a class)
#       a) L1 loss function
#       b) L2 average (conjunctive loss)
#       c) Correlation loss
#    3) Additionally also show the mean of the images in the plot title

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [2]:
# %% Generate data

# 2D Gaussian params
n_gaussians = 1000
img_size    = 91
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

# Vary FWHM smoothly
widths = np.linspace(2,20,n_gaussians)

# Preallocate tensors for images (N,channels,size,size)
images = torch.zeros(n_gaussians,1,img_size,img_size)

# Generate images
for i in range(n_gaussians):

    # Gaussian with random center offset
    c = 1.5*np.random.randn(2)
    G  = np.exp( -( (X-c[0])**2 + (Y-c[1])**2) / widths[i] )

    # Layer some noise
    G  = G + np.random.randn(img_size,img_size)/5

    # Add random bar randomly
    i1 = np.random.choice(np.arange(2,28))
    i2 = np.random.choice(np.arange(2,6))
    if np.random.randn()>0:
        G[i1:i1+i2,] = 1
    else:
        G[:,i1:i1+i2] = 1

    # Add to tensor
    images[i,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(3,7,figsize=(phi*7,7))

for i,ax in enumerate(axs.flatten()):

    pic = np.random.randint(n_gaussians)
    G   = np.squeeze( images[pic,:,:] )

    ax.imshow(G,vmin=-1,vmax=1,cmap='jet')
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.suptitle('Occluded Gaussians')

plt.savefig('figure79_code_challenge_30.png')
plt.show()
files.download('figure79_code_challenge_30.png')


In [23]:
# %% Custom loss function

# L1 loss
class custom_loss_1(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,yHat,y):
        loss = torch.mean( torch.abs(yHat-y) )
        return loss


In [50]:
# %% Custom loss function

# L2 + average loss
class custom_loss_2(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,yHat,y):
        mse  = torch.mean( (yHat-y)**2 )
        avg  = torch.abs( torch.mean(yHat) )
        loss = mse + avg
        return loss


In [40]:
# %% Custom loss function

# Correlation loss (note that you want to optimise for the loss, hence the -)
class custom_loss_3(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,yHat,y):
        mean_yHat = torch.mean(yHat)
        mean_y    = torch.mean(y)
        num       = torch.sum( (yHat-mean_yHat)*(y-mean_y) )
        den       = (torch.numel(y)-1) * torch.std(yHat) * torch.std(y)
        loss      = - num/den
        return loss


In [51]:
# %% Function to generate the model

# Dictionary of available losses
loss_dict = { "custom1": custom_loss_1,
              "custom2": custom_loss_2,
              "custom3": custom_loss_3,
              "custom4": custom_loss_4,
              "custom5": custom_loss_5 }

# Basically for the decoder we do "transpose" convolution
def gen_model(loss_name='custom1'):

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Encoding layer
            self.encoder = nn.Sequential( nn.Conv2d(1,6,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2),
                                          nn.Conv2d(6,4,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2) )

            # Decoding layer
            self.decoder = nn.Sequential( nn.ConvTranspose2d(4,6,3,2),
                                          nn.ReLU(),
                                          nn.ConvTranspose2d(6,1,3,2) )

        def forward(self,x):
            return self.decoder(self.encoder(x))

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function (try all three)
    loss_fun = loss_dict[loss_name]()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [52]:
# %% Function to train the model

def train_model(loss_name):

    # Parameters, model instance, inizialise vars
    num_epochs = 1000
    CNN,loss_fun,optimizer = gen_model(loss_name)

    losses = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # pick a set of images at random
        pics_batch = np.random.choice(n_gaussians,size=32,replace=False)
        X = images[pics_batch,:,:,:]

        # Forward propagation and loss
        yHat = CNN(X)
        loss = loss_fun(yHat,X)

        losses.append(loss.item())

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return losses,CNN


In [58]:
# %% Run the model

losses,CNN = train_model('custom5')


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*5,5))

plt.plot(losses,'-',label='Train')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Model loss (final loss=%.3f)'%losses[-1])

plt.savefig('figure80_code_challenge_30.png')
plt.show()
files.download('figure80_code_challenge_30.png')


In [ ]:
# %% Plotting

pics_batch = np.random.choice(n_gaussians,size=32,replace=False)
X = images[pics_batch,:,:,:]
yHat = CNN(X)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,10,figsize=(1.5*phi*6,6))

for i in range(10):

    G = torch.squeeze( X[i,0,:,:] ).detach()
    O = torch.squeeze( yHat[i,0,:,:] ).detach()

    axs[0,i].imshow(G,vmin=-1,vmax=1,cmap='jet')
    axs[0,i].axis('off')
    axs[0,i].set_title('Input ($\\mu$=%.2f)'%torch.mean(G).item())

    axs[1,i].imshow(O,vmin=-1,vmax=1,cmap='jet')
    axs[1,i].axis('off')
    axs[1,i].set_title('Output ($\\mu$=%.2f)'%torch.mean(O).item())

plt.tight_layout()

plt.savefig('figure81_code_challenge_30.png')
plt.show()
files.download('figure81_code_challenge_30.png')


In [ ]:
# %% Exercise 1
#    The code in this notebook requires "manually" switching between loss functions by (un)commenting. Modify the
#    code so that you can list the name of the loss function as an input to makeTheNet().

# Easy peasy already done with a dictionary


In [48]:
# %% Exercise 2
#    Here's an interesting loss function: minimize the variance of the model's output. Don't worry about comparing
#    to the input image; just set the loss function to be the variance of the output. What do the results look like,
#    and why does this happen?

# Well, loss is zero, but surprise surprise, the images are flat! It happens
# because here we try to minimise the variance in the output, meaning that we
# push all the value to be a constant (hence no variance)

# Variance loss
class custom_loss_4(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,yHat,y):
        loss = torch.var(yHat)
        return loss


In [57]:
# %% Exercise 3
#    What if L2 minimization (MSE) is more important than average minimization? Modify the L2LossAve class so that the
#    average has a weaker influence compared to the L2 loss.

# Yeah even scaling by 0.2 helps a bit, the values are no longer pushed to be
# extremely small by the penalty term

# L2 + average loss (scaled down)
class custom_loss_5(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,yHat,y):
        mse  = torch.mean( (yHat-y)**2 )
        avg  = torch.abs( torch.mean(yHat) )
        loss = mse + avg*0.2
        return loss
